#### This script uses five classifiers, linear,ann,knn,protonet and random_forest to get a final classifier using the slide aggregations.
#### It also outputs results metrics such as Balance accuracy, AUROC etc into a csv in the output folder.

In [1]:
import os
os.getcwd()

'd:\\Aamir Gulzar\\KSA_project2\\Cancer-detection-classifier\\slide_classification'

## Hugging face login

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
from huggingface_hub import login

login(os.environ["HF_TOKEN"])

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful


In [3]:
import torch
import torchvision
from torch.utils.data import DataLoader
import os
from os.path import join as j_
from PIL import Image
import pandas as pd
import numpy as np
import time
import random
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Lambda
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
# loading all packages here to start
# from dataloader import WSIDataset
from eval_patch_features.logistic import eval_linear
from eval_patch_features.ann import eval_ANN
from eval_patch_features.knn import eval_knn
from eval_patch_features.protonet import eval_protonet
from eval_patch_features.r_forest_eval import eval_r_forest
from eval_patch_features.metrics import get_eval_metrics, print_metrics
from utility import calculate_metric_averages, average_confusion_matrices, write_data_in_excel, build_probs_df
import warnings
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Pipeline for TCGA Data

In [ ]:
# =============================================================================
# TCGA-CV  -  4-fold cross-validation on TCGA (canonical name, work order sec. 1)
# =============================================================================
# REWRITTEN 2026-08-04 (work order Phase 1). This cell used to carry its own
# ~90-line WSIDataset, its own DataLoader plumbing and its own train/evaluate
# loop - one of ~6 near-identical copies. It is now a thin driver over the
# shared modules, so a correctness fix lands in one place instead of six.
#
# What changed, and why the numbers this produces supersede the previous ones:
#
#   Task 1.0  Data is loaded once into a single (N, D) tensor and split by index
#             (was: 3 WSIDataset rebuilds per fold per classifier, i.e. every .pt
#             file re-read ~20x). Verified bit-identical to the old loader and
#             metric-identical to 1e-6 - see tools/parity_check.py.
#   Task 1.1  ANN hyperparameters are selected on VALIDATION macro-F1. The old
#             criterion, eval_metrics['ann_macro_f1'], is computed on the TEST
#             set, so the best of 4 configurations was chosen using test
#             performance and reported as a test result.
#   Task 1.2  The ANN checkpoint saved is the configuration actually selected,
#             named fold{f}_ann_{dim}_{h1}_{h2}.pth with a sibling config.json.
#             The old code saved inside the grid loop to a name that did not vary
#             across the grid, so the file on disk was always the LAST entry.
#   Task 1.3  The double softmax is gone (nn.Softmax fed to CrossEntropyLoss).
#             Every ANN checkpoint and ANN result must therefore be regenerated.
#   Task 1.4  A labelled slide with no feature file is now reported, not silently
#             dropped. This cell prints N=413, not 416, and says which 3 are
#             missing. Same reason PAIP is 73 and not 78.
#   Sec. 1b   Out-of-fold probabilities are pooled and tau_TCGA is fitted by
#             Youden's J, written to thresholds_TCGA.json. External experiments
#             (PAIP-EV, SurGen-EV) apply that threshold unchanged; a threshold is
#             never tuned on the external cohort.
#   Sec. 1c.2 No hardcoded D:\ paths. Roots come from config/paths.py, keyed on
#             the MACHINE environment variable, so this runs unmodified on the
#             server.
#
# Previous results are archived to TCGA_Results_ARCHIVED_<date> on first write -
# nothing is deleted or edited in place (rule of engagement #2).

import os
import sys

sys.path.insert(0, os.path.abspath("."))

from config import paths as P
from runners.cv_runner import sweep

# --- what to run -------------------------------------------------------------
# TITAN and PRISM are re-enabled here (work order Task 3.3 - they were commented
# out). TITAN runs only with Conch1_5 and PRISM only with PRISM; those guards
# live in P.is_combination_valid() and are applied by the sweep.
AGGREGATION_METHODS = [
    "Averaging",
    "Caption_based_aggregation",
    "Caption_based_aggregation_15_classes",
    "Tissue_Type_Clustering",
    "TITAN",
    "PRISM",
]
MODEL_NAMES = ["H-Optimus-1", "Conch1_5", "UNI2", "Virchow2", "ConchV1", "PRISM"]
CLASSIFIERS = ["lin", "ann", "knn", "proto", "rf"]
TASK = "MSIH"  # BRAF/KRAS/TP53/CIMP are Phase 5 - deferred, do not enable yet

print(f"machine={P.MACHINE}  base_root={P.BASE_ROOT}")
print(f"results -> {P.results_root('TCGA-CV', 'Averaging', 'UNI2', TASK).parents[2]}")

sweep(
    experiment="TCGA-CV",
    cohort="tcga",
    methods=AGGREGATION_METHODS,
    models=MODEL_NAMES,
    task=TASK,
    model_types=CLASSIFIERS,
    seed=42,
    force=False,          # resume: completed combinations are skipped
    write_thresholds=True,  # TCGA-CV is what defines tau_TCGA (section 1b)
)


<h1>PAIP &mdash; PAIP-CV RETIRED (work order Task 2.2)</h1>

**This section used to run a 4-fold "cross-validation" on PAIP. It has been retired
from the study, 2026-08-04, by owner decision. Only PAIP-IV and PAIP-EV are
reported for the PAIP cohort.** The previous results are preserved at
`slide_classification/PAIP_Results_ARCHIVED_20260804/` (484 files) &mdash; nothing was
deleted, because those numbers may already have been circulated.

### Why it was retired

**The fold construction was broken.** Folds were built by *sequential slicing of
CSV order* &mdash; no shuffle, no stratification:

```python
fold_size = num_total_slides // num_folds
fold_ids  = valid_wsi_ids[start_idx : start_idx + fold_size]
```

The CSV is ordered `training_data_01 … training_data_47, validation_data_01 …`, so
the folds tracked PAIP's own acquisition split rather than a random partition:

| Fold | n | MSI-H | from `training_data_*` | from `validation_data_*` |
|---|---|---|---|---|
| 1 | 18 | 4 | 18 | 0 |
| 2 | 18 | 4 | 18 | 0 |
| 3 | 18 | 5 | 6 | 12 |
| 4 | 19 | 4 | 0 | 19 |

Folds 1 and 2 are entirely the provider's training set; fold 4 is entirely its
validation set. The class balance came out even **by luck of ordering**, not by
design, and is not reproducible under any reordering.

### But the deciding argument is not the bug &mdash; it is that the experiment is strictly dominated

Even with fold construction repaired, PAIP-CV would train on ~36 slides (2 of 4
folds) and test on 18 with ~4 positives. **PAIP-IV trains on 42 and tests on 31.**
PAIP-CV therefore trains on *less* data and tests on *smaller* sets while answering
the same question. An AUROC computed on 14 negatives and 4 positives cannot
separate models &mdash; the per-fold confusion matrices already in
`best_of_all_exps_metric.xlsx` (e.g. `[[14, 0], [1, 3]]`) show this directly.

### What replaces it

| Experiment | Train | Test | Protocol |
|---|---|---|---|
| **PAIP-IV** | PAIP official train, 42 with features (10 MSI-H) | PAIP official test, 31 (7 MSI-H) | single fixed provider split, no averaging |
| **PAIP-EV** | TCGA-FULL (all 413 TCGA slides) | all 73 PAIP slides | external, threshold fixed on TCGA |

**If uncertainty estimates on PAIP are wanted**, PAIP-IV reports bootstrap
confidence intervals over its 31 test slides (1000 resamples, 95% CI on BalAcc and
AUROC). That is the correct tool at this sample size &mdash; not K-fold.

The `PAIP` sheet is dropped from the consolidated workbook and its radar plots are
removed (Phase 4).


In [ ]:
# =============================================================================
# PAIP-CV RETIRED - work order Task 2.2. See the markdown cell above for why.
# =============================================================================
# The ~200-line block that used to live here (its own WSIDataset, its own
# sequential-slicing fold builder, and a 4-fold CV loop) has been removed.
# Nothing here is meant to be run.
#
# PAIP is now covered by two experiments, both driven from runners/:
#
#   PAIP-IV  python -m runners.iv_runner --experiment PAIP-IV
#            provider's official 47/31 split; 42 of the 47 train ids have
#            features. Bootstrap 95% CIs over the 31 test slides.
#            -> built by Task 3.1 (Phase 3)
#
#   PAIP-EV  python -m runners.ev_runner --experiment PAIP-EV
#            TCGA-trained, applied to all 73 PAIP slides at tau_TCGA.
#            -> built by Tasks 3.0/3.2 (Phase 3)
#
# Do not reinstate a PAIP fold loop. If uncertainty on PAIP is wanted, use the
# PAIP-IV bootstrap CIs, not K-fold - at 31 test slides with 7 positives, a
# 4-way split cannot separate models.

raise NotImplementedError(
    "PAIP-CV is retired (work order Task 2.2). Use the PAIP-IV runner "
    "(Task 3.1) or the PAIP-EV runner (Tasks 3.0/3.2) instead. "
    "Previous PAIP-CV results: PAIP_Results_ARCHIVED_20260804/"
)


### TCGA test ann and linear hyperparameters

In [4]:
    #Test ann for pipeline 2
    import pandas as pd
    import itertools
    import os

    # Grid of hyperparameters
    HIDDEN_DIM1_VALUES = [128, 256, 512]
    HIDDEN_DIM2_VALUES = [64, 128, 256]
    MAX_ITER_VALUES = [100, 200, 350, 500, 1000]

    # Storage
    grid_results = []

    # Grid Search over (hidden_dim1, hidden_dim2, max_iter)
    for hidden_dim1, hidden_dim2, max_iter in itertools.product(HIDDEN_DIM1_VALUES, HIDDEN_DIM2_VALUES, MAX_ITER_VALUES):
        print(f"\n🧪 Running CV with HIDDEN_DIM1={hidden_dim1}, HIDDEN_DIM2={hidden_dim2}, MAX_ITER={max_iter}")

        try:
            # Override the ANN tuning loop using these globals
            def custom_train_and_evaluate(fold, train_loader, val_loader, test_loader, model_type='ann'):
                all_feats = lambda loader: zip(*[(x, y, z) for x, y, z in loader])
                train_feats, train_labels, _ = all_feats(train_loader)
                val_feats, val_labels, _ = all_feats(val_loader)
                test_feats, test_labels, all_test_ids = all_feats(test_loader)

                train_feats = torch.cat(train_feats)
                train_labels = torch.cat(train_labels)
                val_feats = torch.cat(val_feats)
                val_labels = torch.cat(val_labels)
                test_feats = torch.cat(test_feats)
                test_labels = torch.cat(test_labels)

                eval_metrics, eval_dump = eval_ANN(
                    fold=fold,
                    train_feats=train_feats,
                    train_labels=train_labels,
                    valid_feats=val_feats,
                    valid_labels=val_labels,
                    test_feats=test_feats,
                    test_labels=test_labels,
                    input_dim=VECTOR_DIM,
                    hidden_dim=hidden_dim1,
                    hidden_dim2=hidden_dim2,
                    max_iter=max_iter,
                    model_save_path=MODEL_SAVE_PATH,
                    verbose=False,
                )
                return eval_metrics, eval_dump, all_test_ids

            # Inject into main loop temporarily
            original_train_and_evaluate = globals()['train_and_evaluate']
            globals()['train_and_evaluate'] = custom_train_and_evaluate

            # Run CV
            results = run_k_fold_cross_validation(
                save_dir=DATA_PATH,
                folds=folds,
                model_type='ann'
            )

            # Restore original function
            globals()['train_and_evaluate'] = original_train_and_evaluate

            # Extract avg metrics
            avg_acc = sum([r.get("ann_bacc", 0) for r in results]) / len(results)
            avg_auc = sum([r.get("ann_auroc", 0) for r in results]) / len(results)

            # Save result
            grid_results.append({
                "HIDDEN_DIM1": hidden_dim1,
                "HIDDEN_DIM2": hidden_dim2,
                "MAX_ITER": max_iter,
                "Avg Accuracy": avg_acc,
                "Avg AUC": avg_auc
            })

        except Exception as e:
            print(f"❌ Error with HIDDEN_DIM1={hidden_dim1}, HIDDEN_DIM2={hidden_dim2}, MAX_ITER={max_iter}: {e}")
            grid_results.append({
                "HIDDEN_DIM1": hidden_dim1,
                "HIDDEN_DIM2": hidden_dim2,
                "MAX_ITER": max_iter,
                "Avg Accuracy": None,
                "Avg AUC": None,
                "Error": str(e)
            })
            globals()['train_and_evaluate'] = original_train_and_evaluate  # Restore in case of exception

    # Save to Excel
    df_grid = pd.DataFrame(grid_results)
    excel_path = os.path.join(OUTPUT_SAVE_PATH, "grid_search_ann_hidden1_hidden2_iter.xlsx")
    df_grid.to_excel(excel_path, index=False)

    # Display best config
    df_sorted = df_grid.sort_values(by="Avg Accuracy", ascending=False)
    best_row = df_sorted.iloc[0]
    print("\n✅ Grid search complete. Best configuration:")
    print(best_row)

    df_grid

    #Test linear for pipeline 2
    import pandas as pd
    import itertools
    import os

    # Grid of hyperparameters for Logistic Regression
    C_VALUES = [0.01, 0.1, 1, 10]
    MAX_ITER_VALUES = [300, 500, 1000]

    # Storage
    grid_results = []

    # Grid Search over (C, max_iter)
    for C, max_iter in itertools.product(C_VALUES, MAX_ITER_VALUES):
        print(f"\n🧪 Running CV with C={C}, MAX_ITER={max_iter}")
        
        try:
            # Override the linear tuning loop using these globals
            def custom_train_and_evaluate(fold, train_loader, val_loader, test_loader, model_type='lin'):
                all_feats = lambda loader: zip(*[(x, y, z) for x, y, z in loader])
                train_feats, train_labels, *_ = all_feats(train_loader)
                val_feats, val_labels, *_ = all_feats(val_loader)
                test_feats, test_labels, all_test_ids = all_feats(test_loader)
                
                train_feats = torch.cat(train_feats)
                train_labels = torch.cat(train_labels)
                val_feats = torch.cat(val_feats)
                val_labels = torch.cat(val_labels)
                test_feats = torch.cat(test_feats)
                test_labels = torch.cat(test_labels)
                
                eval_metrics, eval_dump = eval_linear(
                    fold=fold,
                    train_feats=train_feats,
                    train_labels=train_labels,
                    valid_feats=val_feats,
                    valid_labels=val_labels,
                    test_feats=test_feats,
                    test_labels=test_labels,
                    max_iter=max_iter,
                    save_path=MODEL_SAVE_PATH,
                    C=C,
                    verbose=False,
                )
                return eval_metrics, eval_dump, all_test_ids
            
            # Inject into main loop temporarily
            original_train_and_evaluate = globals()['train_and_evaluate']
            globals()['train_and_evaluate'] = custom_train_and_evaluate
            
            # Run CV
            results = run_k_fold_cross_validation(
                save_dir=DATA_PATH,
                folds=folds,
                model_type='lin'
            )
            
            # Restore original function
            globals()['train_and_evaluate'] = original_train_and_evaluate
            
            # Extract avg metrics (using bacc as key metric)
            avg_bacc = sum([r.get("lin_bacc", 0) for r in results]) / len(results)
            avg_auc = sum([r.get("lin_auroc", 0) for r in results]) / len(results)
            avg_f1 = sum([r.get("lin_macro_f1", 0) for r in results]) / len(results)
            
            # Save result
            grid_results.append({
                "C": C,
                "MAX_ITER": max_iter,
                "Avg BACC": avg_bacc,
                "Avg AUC": avg_auc,
                "Avg F1": avg_f1
            })
            
        except Exception as e:
            print(f"❌ Error with C={C}, MAX_ITER={max_iter}: {e}")
            grid_results.append({
                "C": C,
                "MAX_ITER": max_iter,
                "Avg BACC": None,
                "Avg AUC": None,
                "Avg F1": None,
                "Error": str(e)
            })
            globals()['train_and_evaluate'] = original_train_and_evaluate  # Restore in case of exception

    # Save to Excel
    df_grid = pd.DataFrame(grid_results)
    excel_path = os.path.join(OUTPUT_SAVE_PATH, "grid_search_linear_C_maxiter.xlsx")
    df_grid.to_excel(excel_path, index=False)

    # Display best configuration (sorted by BACC)
    df_sorted = df_grid.sort_values(by="Avg BACC", ascending=False)
    best_row = df_sorted.iloc[0]
    print("\n✅ Linear Grid search complete. Best configuration:")
    print(best_row)
    df_grid


🧪 Running CV with HIDDEN_DIM1=128, HIDDEN_DIM2=64, MAX_ITER=100
Running Fold 1 with model ann...
  Train IDs count: 215, Val IDs count: 98, Test IDs count: 103
  Dataset sizes - Train: 213, Val: 97, Test: 103
ann_acc: 0.7864
ann_bacc: 0.8473
ann_macro_f1: 0.7095
ann_weighted_f1: 0.8154
ann_auroc: 0.9023
ann_conf_matrix: [[67 21]
 [ 1 14]]
Running Fold 2 with model ann...
  Train IDs count: 206, Val IDs count: 112, Test IDs count: 98
  Dataset sizes - Train: 204, Val: 112, Test: 97
ann_acc: 0.8454
ann_bacc: 0.8132
ann_macro_f1: 0.7385
ann_weighted_f1: 0.8609
ann_auroc: 0.8727
ann_conf_matrix: [[72 12]
 [ 3 10]]
Running Fold 3 with model ann...
  Train IDs count: 201, Val IDs count: 103, Test IDs count: 112
  Dataset sizes - Train: 200, Val: 101, Test: 112
ann_acc: 0.8304
ann_bacc: 0.8380
ann_macro_f1: 0.7652
ann_weighted_f1: 0.8447
ann_auroc: 0.8810
ann_conf_matrix: [[76 16]
 [ 3 17]]
Running Fold 4 with model ann...
  Train IDs count: 210, Val IDs count: 103, Test IDs count: 103
  Dat

,C,MAX_ITER,Avg BACC,Avg AUC,Avg F1
0,0.01,300,0.575318,0.845281,0.590929
1,0.01,500,0.575318,0.845281,0.590929
2,0.01,1000,0.575318,0.845281,0.590929
3,0.10,300,0.680520,0.867655,0.713570
4,0.10,500,0.680520,0.867655,0.713570
5,0.10,1000,0.680520,0.867655,0.713570
6,1.00,300,0.720560,0.856302,0.744748
7,1.00,500,0.720560,0.856030,0.744748
8,1.00,1000,0.720560,0.856166,0.744748
9,10.00,300,0.697649,0.840916,0.713317


## Pipeline for TCGA MultiClass classification (full)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import pandas as pd
from typing import Optional, Tuple, List, Dict, Any, Union
from itertools import product
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
)

from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import os
# configs
VECTOR_DIM = 6912  # size of input feature vector, H-optimus-1 = 13824, UNI-2h = 13824, Conchv1_5 = 6912, TITAN = 768, Average_Features = 768
HIDDEN_DIM = 128   # size of ANN hidden layer
BATCH_SIZE = 4
task_name = "TCGA"  # "MSIH", "BRAF", "KRAS", "TP53", "CIMP"
K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\Results\paip_78_labels.csv"
DATA_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\conch_average_fivecrop"
MODEL_SAVE_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\PAIP_Result\Conch1_5" + "\\" + task_name + r"\models"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
OUTPUT_SAVE_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\PAIP_Result\Conch1_5" + "\\" + task_name + r"\Output"
os.makedirs(OUTPUT_SAVE_PATH, exist_ok=True)
# create a excel sheet in the output folder to save the results
EVAL_METRICS_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_eval_metrics.xlsx")
PROBS_ALL_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_probs_all.xlsx")
sheet_name = "baseline"

class ANNMultiClassifier:
    def __init__(
        self,
        input_dim: int = 512,
        hidden_dim1: int = 512,
        hidden_dim2: int = 128,
        num_classes: int = 3,
        max_iter: int = 100,
        verbose: bool = True
    ):
        self.max_iter = max_iter
        self.verbose = verbose
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.input_dim = input_dim
        self.hidden_dim1 = hidden_dim1
        self.hidden_dim2 = hidden_dim2
        self.num_classes = num_classes

        # Model definition (NO softmax here)
        self.model = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dim1),
            nn.ReLU(),
            nn.BatchNorm1d(self.hidden_dim1),
            nn.Dropout(0.3),
            nn.Linear(self.hidden_dim1, self.hidden_dim2),
            nn.ReLU(),
            nn.BatchNorm1d(self.hidden_dim2),
            nn.Dropout(0.3),
            nn.Linear(self.hidden_dim2, self.num_classes)
        ).to(self.device)

        self.loss_func = nn.CrossEntropyLoss()

    def compute_loss(self, preds: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        return self.loss_func(preds, labels)

    def predict_proba(self, feats: torch.Tensor) -> torch.Tensor:
        feats = feats.to(self.device)
        self.model.eval()
        with torch.no_grad():
            logits = self.model(feats)
            return torch.softmax(logits, dim=1)  # Apply softmax here for probabilities

    def fit(
        self,
        train_feats: torch.Tensor,
        train_labels: torch.Tensor,
        val_feats: Optional[torch.Tensor] = None,
        val_labels: Optional[torch.Tensor] = None,
        combine_trainval: bool = False
    ) -> Tuple[List[float], List[float]]:
        train_feats, train_labels = train_feats.to(self.device), train_labels.to(self.device)

        if val_feats is not None:
            val_feats, val_labels = val_feats.to(self.device), val_labels.to(self.device)

        if combine_trainval and val_feats is not None:
            train_feats = torch.cat([train_feats, val_feats], dim=0)
            train_labels = torch.cat([train_labels, val_labels], dim=0)
            val_feats, val_labels = None, None

        optimizer = optim.Adam(self.model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.3, patience=5, verbose=self.verbose
        )

        train_loss_history, val_loss_history = [], []
        best_val_loss = float("inf")
        patience, epochs_no_improve = 20, 0

        for epoch in range(self.max_iter):
            # Training
            self.model.train()
            preds = self.model(train_feats)
            loss = self.compute_loss(preds, train_labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_history.append(loss.item())

            # Validation
            val_loss = None
            if val_feats is not None and not combine_trainval:
                self.model.eval()
                with torch.no_grad():
                    val_preds = self.model(val_feats)
                    val_loss = self.compute_loss(val_preds, val_labels)
                val_loss_history.append(val_loss.item())

                # Early stopping
                if val_loss.item() < best_val_loss:
                    best_val_loss = val_loss.item()
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= patience:
                        if self.verbose:
                            print(f"Early stopping at epoch {epoch}")
                        break

            # Scheduler step
            scheduler.step(val_loss.item() if val_loss is not None else loss.item())

            if self.verbose and epoch % 10 == 0:
                if val_loss is not None:
                    print(f"Epoch {epoch}: Train Loss = {loss:.4f}, Val Loss = {val_loss:.4f}")
                else:
                    print(f"Epoch {epoch}: Train Loss = {loss:.4f}")

        return train_loss_history, val_loss_history


def get_eval_metrics(
    targets_all: Union[List[int], np.ndarray],
    preds_all: Union[List[int], np.ndarray],
    probs_all: Optional[Union[List[float], np.ndarray]] = None,
    get_report: bool = True,
    prefix: str = "",
    roc_kwargs: Dict[str, Any] = {},
) -> Dict[str, Any]:
    """
    Calculate evaluation metrics for binary or multi-class classification.
    """
    bacc = balanced_accuracy_score(targets_all, preds_all)
    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro")
    cls_rep = classification_report(targets_all, preds_all, output_dict=True, zero_division=0)
    conf_matrix = confusion_matrix(targets_all, preds_all)
    
    eval_metrics = {
        f"{prefix}acc": acc,
        f"{prefix}bacc": bacc,
        f"{prefix}macro_f1": macro_f1,
        f"{prefix}weighted_f1": cls_rep["weighted avg"]["f1-score"],
    }
    
    if probs_all is not None:
        try:
            n_classes = probs_all.shape[1]
            if n_classes > 2:
                # Multi-class ROC AUC (One-vs-Rest)
                roc_auc = roc_auc_score(
                    targets_all,
                    probs_all,
                    multi_class="ovr",
                    average="macro",
                    **roc_kwargs
                )
            else:
                # Binary ROC AUC
                roc_auc = roc_auc_score(targets_all, probs_all[:, 1], **roc_kwargs)
            eval_metrics[f"{prefix}auroc"] = roc_auc
        except ValueError:
            eval_metrics[f"{prefix}auroc"] = np.nan
    
    eval_metrics[f"{prefix}conf_matrix"] = conf_matrix
    return eval_metrics


def print_metrics(eval_metrics):
    for key, value in eval_metrics.items():
        if isinstance(value, (int, float)):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")


def eval_ANN(
    fold: int,
    train_feats: torch.Tensor,
    train_labels: torch.Tensor,
    valid_feats: torch.Tensor,
    valid_labels: torch.Tensor,
    test_feats: torch.Tensor,
    test_labels: torch.Tensor,
    input_dim: int = 512,
    hidden_dim: int = 512,
    hidden_dim2: int = 128,
    max_iter: int = 1000,
    num_classes: int = 3,
    prefix: str = "ann_",
    combine_trainval: bool = False,
    model_save_path: str = None,
    verbose: bool = False,
) -> Tuple[dict, dict]:

    if verbose:
        print(f"[Fold {fold}] Train Shape: {train_feats.shape}, Test Shape: {test_feats.shape}")

    classifier = ANNMultiClassifier(
        input_dim=input_dim,
        hidden_dim1=hidden_dim,
        hidden_dim2=hidden_dim2,
        num_classes=num_classes,
        max_iter=max_iter,
        verbose=verbose
    )

    classifier.fit(train_feats, train_labels, valid_feats, valid_labels, combine_trainval)

    if model_save_path is not None:
        os.makedirs(model_save_path, exist_ok=True)
        model_path = os.path.join(model_save_path, f"fold{fold}_trained_ann_model_{input_dim}.pth")
        torch.save(classifier.model.state_dict(), model_path)

    probs_all = classifier.predict_proba(test_feats).cpu().numpy()
    preds_all = np.argmax(probs_all, axis=1)
    targets_all = test_labels.cpu().numpy()

    metrics = get_eval_metrics(
        targets_all=targets_all,
        preds_all=preds_all,
        probs_all=probs_all,
        prefix=prefix
    )

    dump = {
        "preds_all": preds_all,
        "probs_all": probs_all,
        "targets_all": targets_all
    }

    return metrics, dump


def get_feats_labels(loader):
    feats, labels, ids = [], [], []
    for features, lbls, wsi_ids in loader:
        feats.append(features)
        labels.append(lbls)
        if isinstance(wsi_ids, (list, tuple)):
            ids.extend(wsi_ids)
        else:
            ids.append(wsi_ids)
    return torch.cat(feats), torch.cat(labels), ids


# Global variables for tracking averages
avg_fold_metrics = {}
classifier_fold_count = {}


def save_results_to_csv(
    save_path: str,
    classifier: str,
    fold: Union[int, str],
    eval_metrics: Dict[str, Any],
    num_folds: int = 4,
):
    global avg_fold_metrics, classifier_fold_count
    
    os.makedirs(save_path, exist_ok=True)
    results_file_path = os.path.join(save_path, "results.csv")
    
    # Check if file exists, if not create with header
    if not os.path.exists(results_file_path):
        with open(results_file_path, "w") as f:
            f.write("classifier,fold,Accuracy,Balanced Accuracy,Macro-F1,Weighted F1-Score,AUROC\n")
    
    # Append results
    with open(results_file_path, "a") as f:
        f.write(
            f"{classifier},{fold},{eval_metrics['ann_acc']:.4f},{eval_metrics['ann_bacc']:.4f},"
            f"{eval_metrics['ann_macro_f1']:.4f},{eval_metrics['ann_weighted_f1']:.4f},"
            f"{eval_metrics['ann_auroc']:.4f}\n"
        )
        
        # Track metrics for averaging
        if classifier not in avg_fold_metrics:
            avg_fold_metrics[classifier] = {k: 0 for k in eval_metrics.keys()}
            classifier_fold_count[classifier] = 0
            
        for k, v in eval_metrics.items():
            if isinstance(v, (int, float)):
                avg_fold_metrics[classifier][k] += v
        classifier_fold_count[classifier] += 1
        
        # Write average if we've completed all folds
        if classifier_fold_count[classifier] == num_folds:
            avg_results = {k: v / num_folds for k, v in avg_fold_metrics[classifier].items() if isinstance(v, (int, float))}
            f.write(
                f"{classifier},Average,{avg_results['ann_acc']:.4f},{avg_results['ann_bacc']:.4f},"
                f"{avg_results['ann_macro_f1']:.4f},{avg_results['ann_weighted_f1']:.4f},"
                f"{avg_results['ann_auroc']:.4f}\n"
            )
            # Reset tracking
            avg_fold_metrics.pop(classifier)
            classifier_fold_count.pop(classifier)


def main():
    # Load CSV containing fold and wsi_id
    folds_df = pd.read_csv(K_FOLDS_PATH)

    # Ensure correct column names
    assert 'fold' in folds_df.columns and 'WSI_Id' in folds_df.columns, "CSV must have 'fold' and 'WSI_Id' columns"

    # Group wsi_ids by fold
    num_folds = folds_df['fold'].nunique()
    folds = []
    for fold_num in sorted(folds_df['fold'].unique()):
        wsi_ids = folds_df[folds_df['fold'] == fold_num]['WSI_Id'].tolist()
        folds.append(wsi_ids)

    results_per_fold = []
    save_dir = DATA_PATH

    # Grid Search parameters for ANN
    param_grid = {
        'hidden_dim1': [128, 256, 512],
        'hidden_dim2': [64, 128, 256],
        'max_iter': [300, 500, 1000]
    }

    for i in range(num_folds):
        # Define test and validation folds
        test_ids = folds[i]
        val_ids = folds[(i + 1) % num_folds]

        # Use remaining folds as training
        train_ids = []
        for j in range(num_folds):
            if j != i and j != (i + 1) % num_folds:
                train_ids.extend(folds[j])
                
        print(f"Running Fold {i + 1} with model ann...")
        
        # Create datasets and loaders
        train_dataset = WSIDataset(save_dir, train_ids)
        val_dataset = WSIDataset(save_dir, val_ids)
        test_dataset = WSIDataset(save_dir, test_ids)
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        train_feats, train_labels, _ = get_feats_labels(train_loader)
        val_feats, val_labels, _ = get_feats_labels(val_loader)
        test_feats, test_labels, all_test_ids = get_feats_labels(test_loader)

        # Grid Search for ANN
        best_metrics = None
        best_dump = None
        best_params = None

        for h1, h2, max_iter in product(param_grid['hidden_dim1'], param_grid['hidden_dim2'], param_grid['max_iter']):
            eval_metrics, eval_dump = eval_ANN(
                fold=i,
                train_feats=train_feats,
                train_labels=train_labels,
                valid_feats=val_feats,
                valid_labels=val_labels,
                test_feats=test_feats,
                test_labels=test_labels,
                input_dim=VECTOR_DIM,
                hidden_dim=h1,
                hidden_dim2=h2,
                max_iter=max_iter,
                model_save_path=MODEL_SAVE_PATH,
                verbose=True,
            )

            # Use balanced_accuracy as the primary selection metric
            if not best_metrics or eval_metrics['ann_bacc'] > best_metrics['ann_bacc']:
                best_metrics = eval_metrics
                best_dump = eval_dump
                best_params = (h1, h2, max_iter)

        print(f"Best ANN params for fold {i}: "
              f"h1={best_params[0]}, h2={best_params[1]}, max_iter={best_params[2]}")
        
        # Save results
        save_results_to_csv(
            save_path=MODEL_SAVE_PATH,
            classifier="ann",
            fold=i+1,
            eval_metrics=best_metrics,
            num_folds=num_folds
        )
        
        results_per_fold.append({
            'fold': i+1,
            'metrics': best_metrics,
            'dump': best_dump,
            'best_params': best_params
        })

    print("Training completed for all folds!")
    return results_per_fold


if __name__ == "__main__":
    main()

## Pipeline for TCGA → PAIP External Validation

Loads the classifiers already trained/saved by the **TCGA pipeline** above (per fold, per model type) and evaluates them zero-shot on **PAIP** slide-aggregation features. Follows the same `AGGREGATION_METHODS` × `MODEL_NAMES` × `task_names` sweep, and writes results with `write_data_in_excel()` in the same `Metric | Fold1..Fold4 | AvgFolds` layout used by the TCGA/PAIP training pipelines, including the `<model>_conf_matrix` row.

**How "Fold" works here:** PAIP itself is not split into folds. Each of the 4 TCGA fold checkpoints is run over the *entire* PAIP dataset, so `Fold1..Fold4` are 4 full-dataset runs from 4 different trained models, and `AvgFolds` is their average — the same convention `AvgFolds` already uses in the TCGA/PAIP training pipelines above.

**Confusion matrices:** collected for every fold and averaged across folds with `average_confusion_matrices()` (previously left blank for external validation). The average is written straight into the `<model>_conf_matrix` row's `AvgFolds` cell in the eval-metrics workbook — no separate confusion-matrix file is created.

In [5]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib
from typing import Dict, List, Tuple, Optional
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

# =============================================================================
# EXTERNAL VALIDATION: models trained on TCGA -> evaluated (zero-shot) on PAIP
# Mirrors the TCGA / PAIP pipeline cells above: same AGGREGATION_METHODS /
# MODEL_NAMES loop, same TASK_PREFIXES, same Excel output convention via
# write_data_in_excel(), and the same eval_metrics_df layout
# (Metric | Fold1..Fold4 | AvgFolds) including a "<model>_conf_matrix" row.
#
# FIX vs. the original standalone validation script: confusion matrices are
# now (1) collected for every fold and (2) averaged across folds using the
# same average_confusion_matrices() utility used by the TCGA/PAIP training
# cells (previously this was left blank for external validation). The average
# is written into the "<model>_conf_matrix" row's AvgFolds column of the
# existing eval_metrics workbook - no separate confusion-matrix file is created.
#
# FIX: ANN architecture is now inferred directly from each checkpoint's
# tensor shapes instead of being read from a fold{fold}_ann_best_params.json
# file, since that json could drift out of sync with the actual saved .pth.
# ==============================================================================

# --- DYNAMIC CONFIGURATIONS (same convention as TCGA / PAIP cells) ---
AGGREGATION_METHODS = ["Averaging", "Caption_based_aggregation", "Caption_based_aggregation_15_classes", "Tissue_Type_Clustering"]  # ["Averaging", "Caption_based_aggregation", "Caption_based_aggregation_15_classes", "Tissue_Type_Clustering", "TITAN", "PRISM"]
MODEL_NAMES = ["H-Optimus-1", "Conch1_5", "UNI2", "Virchow2", "ConchV1"]  # H-Optimus-1 , Conch1_5 , UNI2 , Virchow2 , ConchV1 , PRISM
RESULT_FOLDER = "TCGA_PAIP_EV_Results"

# Define Base Dimensions for each model
MODEL_BASE_DIMS = {
    "H-Optimus-1": 1536,
    "UNI2": 1536,
    "Conch1_5": 768,
    "Virchow2": 2560,
    "ConchV1": 512,
    "PRISM": 1280
}

# Define Multipliers for each aggregation method
AGG_MULTIPLIERS = {
    "Caption_based_aggregation": 14,
    "Caption_based_aggregation_15_classes": 15,
    "Tissue_Type_Clustering": 9,
    "Averaging": 1,
    "TITAN": 1,
}

TASK_PREFIXES = {
    "MSIH": "1-MSIH",
    "BRAF": "2-BRAF",
    "KRAS": "3-KRAS",
    "TP53": "4-TP53",
    "CIMP": "5-CIMP"
}

BATCH_SIZE = 4
task_names = ["MSIH"]  # "MSIH", "BRAF", "KRAS", "TP53", "CIMP"

BASE_ROOT = r"D:\Aamir Gulzar\KSA_project2"
PROJECT_ROOT = os.path.join(BASE_ROOT, "Cancer-detection-classifier", "slide_classification")

# PAIP ground truth (external test set labels)
GROUND_TRUTH_PATH = os.path.join(BASE_ROOT, "paip_data", "labels", "paip_78_labels.csv")

sheet_name = "baseline"

# All TCGA cross-validation folds whose saved classifiers we test on PAIP.
# NOTE: PAIP itself is NOT split into folds - the entire PAIP dataset is
# scored by each of the 4 fold-specific TCGA models in turn (an ensemble of
# 4 checkpoints, not a 4-way data split), and "AvgFolds" below is the average
# of those 4 full-dataset runs - the same convention "AvgFolds" already uses
# in the TCGA / PAIP training pipelines above.
FOLDS = [0, 1, 2, 3]

# Filename templates for the classifiers saved by the TCGA training pipeline.
# Keys match the short model_type names used everywhere else in this notebook
# ('lin', 'ann', 'knn', 'proto', 'rf') so metric rows line up with the TCGA /
# PAIP pipelines above.
CLASSIFIER_TEMPLATES = {
    'lin':   'fold{fold}_logistic_regression.pkl',
    'ann':   'fold{fold}_trained_ann_model_{input_dim}.pth',
    'knn':   'fold{fold}_knn_model.pkl',
    'proto': 'fold{fold}_protonet_model.pkl',
    'rf':    'fold{fold}_random_forest.pkl',
}
CLASSIFIERS_TO_RUN = ['lin', 'ann', 'knn', 'proto', 'rf']

metric_indices = {
    'acc': 0,
    'bacc': 1,
    'macro_f1': 2,
    'weighted_f1': 3,
    'auroc': 4
}


class ANNClassifier(nn.Module):
    """Must match the architecture used during TCGA training."""
    def __init__(self, input_dim: int, hidden_dim: int = 256,
                 hidden_dim2: int = 128, num_classes: int = 2):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim2, num_classes),
            nn.Softmax(dim=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


# ============================================
# LABEL & GROUND TRUTH HELPERS
# ============================================
def parse_label(raw) -> Optional[int]:
    """
    Convert any label string/int to binary.
      nonMSIH / 0 / mss  ->  0
      MSIH    / 1 / msi  ->  1
    """
    if raw is None:
        return None
    s = str(raw).strip().lower()
    if s in ('0', 'nonmsih', 'non-msih', 'mss'):
        return 0
    if s in ('1', 'msih', 'msi-h', 'msih_c', 'msi'):
        return 1
    try:
        return int(float(s))
    except ValueError:
        return None


def load_ground_truth(csv_path: str) -> pd.DataFrame:
    """Load CSV and add normalised _wsi_id and _bin_label columns."""
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    print(f"Ground truth: {len(df)} rows | columns: {df.columns.tolist()}")

    id_col = 'WSI_Id' if 'WSI_Id' in df.columns else df.columns[0]

    lbl_col = None
    for cand in ('label', 'label_desc', 'Label', 'MSI_status'):
        if cand in df.columns:
            lbl_col = cand
            break
    if lbl_col is None:
        raise ValueError(f"No label column found in {df.columns.tolist()}")

    df['_wsi_id'] = df[id_col].astype(str).str.strip()
    df['_bin_label'] = df[lbl_col].apply(parse_label)

    bad = df['_bin_label'].isna().sum()
    if bad:
        print(f"  {bad} rows with unparseable labels - will be skipped")

    return df


def get_label(wsi_id: str, gt_df: pd.DataFrame) -> Optional[int]:
    """
    Look up binary label for a WSI id with three fallback strategies:
      1. Exact match on _wsi_id
      2. Partial / substring match
      3. Extract MSIH / nonMSIH directly from the filename stem
    """
    row = gt_df[gt_df['_wsi_id'] == wsi_id]
    if not row.empty:
        return row['_bin_label'].values[0]

    for _, r in gt_df.iterrows():
        if r['_wsi_id'] in wsi_id or wsi_id in r['_wsi_id']:
            return r['_bin_label']

    lower = wsi_id.lower()
    if '_nonmsih' in lower:
        return 0
    if '_msih' in lower:
        return 1

    return None


# ============================================
# FEATURE & MODEL LOADING HELPERS
# ============================================
def load_features(feature_dir: str) -> Tuple[Dict[str, torch.Tensor], List[str]]:
    """Load all .pt files from a directory. Flattens multi-dim tensors."""
    features_dict: Dict[str, torch.Tensor] = {}
    wsi_ids: List[str] = []

    if not os.path.isdir(feature_dir):
        print(f"  Directory not found: {feature_dir}")
        return features_dict, wsi_ids

    files = [f for f in os.listdir(feature_dir) if f.endswith('.pt')]
    print(f"  Found {len(files)} .pt files in: {os.path.basename(feature_dir)}")

    for fname in files:
        wsi_id = os.path.splitext(fname)[0]
        try:
            feat = torch.load(os.path.join(feature_dir, fname), map_location='cpu')
            if feat.dim() > 1:
                feat = feat.flatten()  # (N_crops, D) -> (N_crops * D,)
            features_dict[wsi_id] = feat.float()
            wsi_ids.append(wsi_id)
        except Exception as e:
            print(f"  Could not load {fname}: {e}")

    return features_dict, wsi_ids


def load_model(classifier_name: str, model_path: str, input_dim: int):
    """Load either a sklearn (.pkl) or PyTorch ANN (.pth) model."""
    if classifier_name == 'ann':
        state = torch.load(model_path, map_location='cpu')
        if isinstance(state, dict) and 'model_state_dict' in state:
            state = state['model_state_dict']

        # Infer architecture straight from the checkpoint's tensor shapes so
        # it always matches, regardless of any external params file.
        hidden_dim1 = state['0.weight'].shape[0]
        hidden_dim2 = state['4.weight'].shape[0]

        m = ANNClassifier(input_dim, hidden_dim=hidden_dim1, hidden_dim2=hidden_dim2)
        m.model.load_state_dict(state)
        m.eval()
        return m
    else:
        loaded = joblib.load(model_path)
        if isinstance(loaded, dict):
            if 'model' in loaded:
                return loaded['model']
            elif 'classifier' in loaded:
                return loaded['classifier']
            elif 'prototypes' in loaded and 'labels_proto' in loaded:
                # Protonet saved as a dict of prototypes - handled in predict()
                return loaded
        return loaded


def predict(classifier_name: str, model, feat: torch.Tensor):
    """
    Run inference for a single sample.
    Returns (pred: int, proba: np.ndarray shape (2,) or None)
    """
    if classifier_name == 'ann':
        x = feat.unsqueeze(0) if feat.dim() == 1 else feat
        with torch.no_grad():
            out = model(x)
            proba = out.numpy()[0]
            pred = int(out.argmax(dim=1).item())
        return pred, proba

    elif classifier_name == 'proto' and isinstance(model, dict):
        if 'prototypes' in model and 'labels_proto' in model:
            prototypes = model['prototypes']
            labels_proto = model['labels_proto']

            if isinstance(prototypes, torch.Tensor):
                prototypes = prototypes.numpy()
            if isinstance(labels_proto, torch.Tensor):
                labels_proto = labels_proto.numpy()

            feat_np = feat.numpy().reshape(1, -1)
            distances = np.linalg.norm(feat_np - prototypes, axis=1)

            closest_idx = np.argmin(distances)
            pred = int(labels_proto[closest_idx])

            inv_distances = 1.0 / (distances + 1e-8)
            proba_raw = np.zeros(2)
            for i, label in enumerate(labels_proto):
                proba_raw[int(label)] += inv_distances[i]

            proba = proba_raw / proba_raw.sum() if proba_raw.sum() > 0 else np.array([0.5, 0.5])
            return pred, proba
        else:
            raise ValueError(f"Protonet model dict missing required keys: {model.keys()}")

    else:
        X = feat.numpy().reshape(1, -1)
        pred = int(model.predict(X)[0])
        try:
            proba = model.predict_proba(X)[0]
        except Exception:
            proba = None
        return pred, proba


# ============================================
# METRICS HELPER
# FIX: confusion_matrix is returned as a numpy array (not just a list) so it
# can be fed straight into average_confusion_matrices(), matching how the
# TCGA / PAIP pipeline cells above already average confusion matrices across
# folds.
# ============================================
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                     y_proba: Optional[np.ndarray]) -> Dict:
    """
    Compute Accuracy, Balanced Accuracy, F1 (macro/weighted), AUROC and the
    raw confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    metrics = {
        'acc': accuracy_score(y_true, y_pred),
        'bacc': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
        'auroc': None,
        'conf_matrix': cm,  # numpy array, shape (2, 2) -> [[tn, fp], [fn, tp]]
    }

    if y_proba is not None and len(np.unique(y_true)) == 2:
        try:
            pos = y_proba[:, 1] if y_proba.ndim == 2 else y_proba
            metrics['auroc'] = roc_auc_score(y_true, pos)
        except Exception:
            pass

    return metrics


# ===========================================================================
# OUTER LOOP: same (aggregation_method, model_name) sweep as the TCGA / PAIP
# training pipelines above - we simply point each combination's saved TCGA
# models at the matching PAIP feature directory.
# ===========================================================================
for AGGREGATION_METHOD in AGGREGATION_METHODS:
    for MODEL_NAME in MODEL_NAMES:

        if AGGREGATION_METHOD == "TITAN" and MODEL_NAME != "Conch1_5":
            continue
        if AGGREGATION_METHOD == "PRISM" and MODEL_NAME != "PRISM":
            continue

        print(f"\n{'='*70}")
        print(f"  External Validation | Aggregation: {AGGREGATION_METHOD} | Model: {MODEL_NAME}")
        print(f"{'='*70}\n")

        if AGGREGATION_METHOD == "PRISM":
            PAIP_DATA_PATH = os.path.join(BASE_ROOT, "paip_data", "slide_aggregation", "PRISM")
        else:
            PAIP_DATA_PATH = os.path.join(BASE_ROOT, "paip_data", "slide_aggregation", AGGREGATION_METHOD, MODEL_NAME)

        for task_name in task_names:
            folder_task_name = TASK_PREFIXES.get(task_name, task_name)

            if AGGREGATION_METHOD == "PRISM":
                TCGA_MODELS_PATH = os.path.join(PROJECT_ROOT, "TCGA_Results", "PRISM", folder_task_name, "models")
            else:
                TCGA_MODELS_PATH = os.path.join(PROJECT_ROOT, "TCGA_Results", AGGREGATION_METHOD, MODEL_NAME, folder_task_name, "models")

            RESULT_ROOT = os.path.join(PROJECT_ROOT, RESULT_FOLDER, AGGREGATION_METHOD, MODEL_NAME)
            OUTPUT_SAVE_PATH = os.path.join(RESULT_ROOT, folder_task_name, "Output")
            os.makedirs(OUTPUT_SAVE_PATH, exist_ok=True)

            EVAL_METRICS_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "external_validation_eval_metrics.xlsx")
            PROBS_ALL_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "external_validation_probs_all.xlsx")

            print(f"Processing Task: {task_name} | TCGA models: {TCGA_MODELS_PATH}")
            print(f"                 PAIP features: {PAIP_DATA_PATH}")

            try:
                gt_df = load_ground_truth(GROUND_TRUTH_PATH)
            except Exception as e:
                print(f"  Failed to load ground truth: {e}")
                continue

            features_dict, wsi_ids = load_features(PAIP_DATA_PATH)
            if not features_dict:
                print("  No PAIP features found - skipping this combination")
                continue

            input_dim = next(iter(features_dict.values())).shape[0]
            print(f"  Loaded {len(features_dict)} PAIP slides | Input dim: {input_dim}")

            wsi_labels = {}
            for wsi_id in wsi_ids:
                lbl = get_label(wsi_id, gt_df)
                if lbl is not None:
                    wsi_labels[wsi_id] = lbl
            print(f"  Valid PAIP WSIs with labels: {len(wsi_labels)} / {len(wsi_ids)}")

            eval_metrics__for_excel = []
            probs_all_for_excel = None

            for model in CLASSIFIERS_TO_RUN:
                print(f"\n\n ********* External validation with model: {model} ********* \n\n")

                per_fold_metrics = {m: [None] * len(FOLDS) for m in metric_indices.keys()}
                per_fold_conf_matrices = [None] * len(FOLDS)
                model_probs_rows = []

                for fold_idx, fold in enumerate(FOLDS):
                    clf_template = CLASSIFIER_TEMPLATES[model]
                    model_filename = clf_template.format(fold=fold, input_dim=input_dim)
                    model_path = os.path.join(TCGA_MODELS_PATH, model_filename)

                    if not os.path.exists(model_path):
                        print(f"  Fold {fold + 1}: TCGA model not found ({model_filename}) - skipping")
                        continue

                    try:
                        loaded_model = load_model(model, model_path, input_dim)
                    except Exception as e:
                        print(f"  Fold {fold + 1}: failed to load model - {e}")
                        continue

                    y_true_list, y_pred_list, y_proba_list, wsi_id_list = [], [], [], []
                    for wsi_id, feat in features_dict.items():
                        if wsi_id not in wsi_labels:
                            continue
                        try:
                            pred, proba = predict(model, loaded_model, feat)
                        except Exception:
                            continue
                        y_true_list.append(wsi_labels[wsi_id])
                        y_pred_list.append(pred)
                        y_proba_list.append(proba)
                        wsi_id_list.append(wsi_id)

                    if not y_true_list:
                        print(f"  Fold {fold + 1}: no predictions produced - skipping")
                        continue

                    y_true = np.array(y_true_list)
                    y_pred = np.array(y_pred_list)
                    y_proba = np.array(y_proba_list) if y_proba_list[0] is not None else None

                    fold_metrics = compute_metrics(y_true, y_pred, y_proba)
                    print_metrics({f"{model}_{k}": v for k, v in fold_metrics.items() if k != 'conf_matrix'})

                    for m in ['acc', 'bacc', 'macro_f1', 'weighted_f1']:
                        per_fold_metrics[m][fold_idx] = fold_metrics[m]
                    if fold_metrics['auroc'] is not None:
                        per_fold_metrics['auroc'][fold_idx] = fold_metrics['auroc']

                    per_fold_conf_matrices[fold_idx] = fold_metrics['conf_matrix']

                    for i, wsi_id in enumerate(wsi_id_list):
                        model_probs_rows.append({
                            "Fold": fold + 1,
                            "WSI_ID": wsi_id,
                            "Target": int(y_true[i]),
                            f"{model}_Pred": int(y_pred[i]),
                            f"{model}_Prob": round(float(y_proba[i, 1]), 4) if y_proba is not None else None,
                        })

                # === Merge this model's predictions into the combined probs table ===
                model_df = pd.DataFrame(model_probs_rows) if model_probs_rows else pd.DataFrame(
                    columns=["Fold", "WSI_ID", "Target", f"{model}_Pred", f"{model}_Prob"])
                if probs_all_for_excel is None:
                    probs_all_for_excel = model_df
                else:
                    probs_all_for_excel = pd.merge(probs_all_for_excel, model_df,
                                                     on=["Fold", "WSI_ID", "Target"], how="outer")

                # === Average metrics across folds ===
                average_results = {}
                for m, values in per_fold_metrics.items():
                    valid_values = [v for v in values if v is not None]
                    if valid_values:
                        average_results[f"{model}_{m}"] = float(np.mean(valid_values))

                # === FIX: average confusion matrix across folds (previously left blank
                # for external validation) - reuses the same average_confusion_matrices()
                # utility already used by the TCGA / PAIP pipeline cells above.
                valid_conf_matrices = [cm for cm in per_fold_conf_matrices if cm is not None]
                avg_conf_matrix = average_confusion_matrices(valid_conf_matrices) if valid_conf_matrices else None

                print("\n\n Average external-validation results across folds:")
                for metric, value in average_results.items():
                    print(f"{metric}: {value:.4f}")
                if avg_conf_matrix is not None:
                    print(f"{model}_conf_matrix (avg): {avg_conf_matrix}")

                # Per-metric rows -> eval_metrics_df (Metric | Fold1..Fold4 | AvgFolds)
                for m in metric_indices.keys():
                    row = [f"{model}_{m}"]
                    for v in per_fold_metrics[m]:
                        row.append(v if v is not None else 'N/A')
                    row.append(average_results.get(f"{model}_{m}", 'N/A'))
                    eval_metrics__for_excel.append(row)

                # Confusion-matrix row (per fold string + averaged matrix) -> eval_metrics_df
                row = [f"{model}_conf_matrix"]
                for cm in per_fold_conf_matrices:
                    row.append(str(cm.tolist()) if cm is not None else 'N/A')
                row.append(str(avg_conf_matrix) if avg_conf_matrix is not None else 'N/A')
                eval_metrics__for_excel.append(row)

            eval_metrics_df = pd.DataFrame(eval_metrics__for_excel,
                                            columns=["Metric", "Fold1", "Fold2", "Fold3", "Fold4", "AvgFolds"])
            write_data_in_excel(EVAL_METRICS_EXCEL, eval_metrics_df, sheet_name=sheet_name)
            write_data_in_excel(PROBS_ALL_EXCEL, probs_all_for_excel, sheet_name=sheet_name)

            print(f"\n✅ Saved eval metrics (incl. averaged confusion matrix) -> {EVAL_METRICS_EXCEL}")
            print(f"✅ Saved per-slide probs                                 -> {PROBS_ALL_EXCEL}")

print("\n✅ TCGA -> PAIP External Validation Complete!")


  External Validation | Aggregation: Averaging | Model: H-Optimus-1

Processing Task: MSIH | TCGA models: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\TCGA_Results\Averaging\H-Optimus-1\1-MSIH\models
                 PAIP features: D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Averaging\H-Optimus-1
Ground truth: 78 rows | columns: ['WSI_Id', 'label']
  Found 73 .pt files in: H-Optimus-1
  Loaded 73 PAIP slides | Input dim: 1536
  Valid PAIP WSIs with labels: 73 / 73


 ********* External validation with model: lin ********* 


lin_acc: 0.7808
lin_bacc: 0.5294
lin_macro_f1: 0.4931
lin_weighted_f1: 0.6971
lin_auroc: 0.8624
lin_acc: 0.7808
lin_bacc: 0.5294
lin_macro_f1: 0.4931
lin_weighted_f1: 0.6971
lin_auroc: 0.8613
lin_acc: 0.7808
lin_bacc: 0.5294
lin_macro_f1: 0.4931
lin_weighted_f1: 0.6971
lin_auroc: 0.8582
lin_acc: 0.8082
lin_bacc: 0.5882
lin_macro_f1: 0.5944
lin_weighted_f1: 0.7518
lin_auroc: 0.8687


 Average external-validation results